# Frozen BGE-M3 preference rankers

Fit root and all-comment neural rankers using the complete production BGE-M3 comment embeddings plus exactly the scalar predictors in the shared XGBoost feature contract. BGE-M3 remains frozen; only the text projection, metadata projection, and separate audience/curator heads are trained.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

from commentgap_analysis.neural_ranking import default_recipe, run_neural_ranker_workflow

MODEL_DATA_ROOT = Path(os.getenv("COMMENTGAP_MODEL_DATA_ROOT", "model_output/selection_2025/model_data"))
DATA_ROOT = Path(os.getenv("COMMENTGAP_DATA_ROOT", "data/scrape_2025"))
EMBEDDING_ROOT = Path(os.getenv("COMMENTGAP_EMBEDDING_ROOT", "model_output/selection_2025/embeddings"))
OUTPUT_ROOT = Path(os.getenv("COMMENTGAP_FROZEN_BGE_ROOT", "model_output/selection_2025/neural_rankers/frozen_bge_m3"))
DEVICE = os.getenv("COMMENTGAP_NEURAL_DEVICE", "auto")
BOOTSTRAP_DRAWS = int(os.getenv("COMMENTGAP_BOOTSTRAP_DRAWS", "1000"))
PROGRESS_EVERY_STORIES = int(os.getenv("COMMENTGAP_NEURAL_PROGRESS_EVERY_STORIES", "100"))
FORCE = os.getenv("COMMENTGAP_NEURAL_FORCE_RECOMPUTE", "0").lower() in {"1", "true", "yes"}
recipe = default_recipe("frozen_bge")
parameters = json.loads((MODEL_DATA_ROOT / "preprocessing_parameters.json").read_text())
assert parameters["version"] == 4, "Force-rerun 06A2 shared preprocessing first"
recipe

NeuralTrainingRecipe(approach='frozen_bge', model_id='BAAI/bge-m3', revision='5617a9f61b028005a4858fdac845db406aefb181', max_length=512, negatives_per_positive=4, text_projection_dim=256, metadata_projection_dim=128, head_hidden_dim=128, dropout=0.1, max_epochs=30, patience=3, pair_batch_size=1024, gradient_accumulation=1, encoder_learning_rate=0.0, head_learning_rate=0.0003, weight_decay=0.0001, gradient_clip=1.0, candidate_batch_size=8192, seed=20260813)

## Five-fold development training and sealed test scoring

In [ ]:
results = run_neural_ranker_workflow(
    MODEL_DATA_ROOT, DATA_ROOT, OUTPUT_ROOT,
    approach="frozen_bge", embedding_root=EMBEDDING_ROOT,
    device=DEVICE, bootstrap_draws=BOOTSTRAP_DRAWS,
    progress_every_stories=PROGRESS_EVERY_STORIES,
    force_recompute=FORCE, recipe=recipe,
)
results

In [3]:
required = {"test_scores_wide.parquet", "test_scores_long.parquet", "test_article_metrics.parquet", "test_metric_summary.parquet", "test_tie_sensitivity_metrics.parquet", "model_manifest.json", "workflow_cache.json"}
for scope in ("root", "all"):
    missing = sorted(name for name in required if not (OUTPUT_ROOT / scope / name).exists())
    assert not missing, (scope, missing)
    manifest = json.loads((OUTPUT_ROOT / scope / "model_manifest.json").read_text())
    assert manifest["reported_scores"] == "sealed_paper2_test"
    assert manifest["test_used_for_selection"] is False
{scope: json.loads((OUTPUT_ROOT / scope / "model_manifest.json").read_text()) for scope in ("root", "all")}

{'root': {'approach': 'frozen_bge',
  'development_articles': 2559,
  'device': {'requested': 'auto', 'resolved': 'cuda'},
  'embedding_store': 'model_output/selection_2025/embeddings/model=BAAI__bge-m3--d790e737/build=de5b3016fb2f-010a7cc75a88',
  'environment': {'packages': {'accelerate': None,
    'bitsandbytes': None,
    'numpy': '2.4.2',
    'pandas': '3.0.1',
    'peft': None,
    'pyarrow': '25.0.1',
    'sentence-transformers': '5.2.2',
    'torch': '2.10.0',
    'transformers': '5.2.0'},
   'platform': 'Linux-6.14.0-32-generic-x86_64-with-glibc2.39'},
  'features': ['log_words',
   'sentiment_positive',
   'sentiment_negative',
   'toxicity_probability',
   'lexdiv_length_adjusted',
   'reading_level_length_adjusted',
   'url_present',
   'article_similarity_top3',
   'novelty_prior_roots_model',
   'log_hours_since_article',
   'prior_reply_composition',
   'discussion_pace',
   'vienna_overnight',
   'vienna_weekday_shoulder_evening',
   'vienna_weekend_day_evening',
   'lo